# Laboratorio 4 — Regresión Avanzada con Ingeniería de Características
## Analítica de Datos · Universidad de Antioquia · Instituto de Matemáticas
**Profesor:** Duván Cataño

---

**Objetivo:** Desarrollar un pipeline completo de modelado para predecir el precio nocturno
de alojamientos Airbnb en Ciudad de México, maximizando el R² en el conjunto de prueba
sin overfitting.

| Técnica | Justificación |
|---|---|
| **Log-transform del target** | `price` tiene skewness >15; el log normaliza la distribución y mejora todos los modelos |
| **Ingeniería de 28 características** | Actividad, ratios, log-transforms, texto del campo `name` y popularidad del barrio |
| **Tratamiento de outliers (IQR×3 ∩ p99)** | Elimina precios extremos sin descartar variabilidad legítima |
| **XGBoost tuned** | Gradient boosting con RandomizedSearchCV (15 iter, K=3) — mejor balance calidad/velocidad |
| **5 modelos comparados en CV** | Regresión Lineal, Ridge, Random Forest, XGBoost, LightGBM |

## 0. Comprensión del Problema y Criterio de Éxito

- **Variable objetivo:** `price` — precio por noche en MXN de un alojamiento Airbnb en CDMX.
- **Tipo de regresión:** Regresión continua multivariada.
- **Criterio de éxito:** R² ≥ 0.75 en conjunto de prueba, con Gap R²(train−test) < 0.10.

### Diagnóstico inicial del target

El precio de Airbnb en CDMX tiene distribución **altamente asimétrica a la derecha**:
- Mediana: ~1,039 MXN/noche · Media: ~1,792 MXN/noche · Máximo: 900,000 MXN/noche
- **Skewness > 15** — los modelos de regresión ordinaria asumen errores homocedásticos,
  lo que no se cumple con esta distribución.

**Decisión técnica:** modelar `log(price)` como target.
- Normaliza la distribución (skewness ≈ 0.4 tras el log)
- Los árboles de boosting encuentran mejores puntos de corte
- Las métricas finales se reportan en **escala original (MXN)** via `exp(predicción)`

> `neighbourhood_group` está completamente vacío en este dataset → excluida. Se usa
> `neighbourhood` (alcaldía de CDMX) con 16 categorías como feature geográfica.

## 1. Importaciones y Configuración de Rutas

Se utiliza `pathlib.Path` con rutas absolutas derivadas de la ubicación del notebook,
eliminando dependencia del directorio de trabajo desde el que se lance Jupyter.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from pathlib import Path
from datetime import datetime

from scipy import stats

from sklearn.model_selection import (
    train_test_split, KFold, cross_validate,
    RandomizedSearchCV, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# ── Rutas absolutas derivadas de la ubicación del notebook ────────────────
_nb_dir    = Path(os.path.abspath(''))
PROJECT_DIR = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
DATA_PATH   = PROJECT_DIR / 'data' / 'raw' / 'dataset_regresion_listings.csv'
REPORTS_DIR = PROJECT_DIR / 'reports' / 'lab4'
MODELS_DIR  = PROJECT_DIR / 'models'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('=' * 60)
print('  LAB 4 — REGRESIÓN — Analítica de Datos — UdeA')
print('=' * 60)
print(f'  PROJECT_DIR : {PROJECT_DIR}')
print(f'  DATA_PATH   : {DATA_PATH}')
print(f'  xgboost     : {XGBRegressor.__module__.split(".")[0]} OK')
print(f'  lightgbm    : {LGBMRegressor.__module__.split(".")[0]} OK')

## 2. Carga de Datos y EDA Inicial

El dataset contiene listados de Airbnb de Ciudad de México. Se realizan tres pasos:
1. Inspección estructural (shape, tipos, nulos)
2. Análisis de la distribución del target `price`
3. Visualización comparativa: distribución original vs log-transform

In [ ]:
print('[1/9] Cargando dataset...')
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'  Filas: {df_raw.shape[0]:,}  |  Columnas: {df_raw.shape[1]}')
print(f'  Columnas: {list(df_raw.columns)}')

print('\n--- Muestra de datos ---')
display(df_raw.head(3))

# Análisis de valores faltantes
info = pd.DataFrame({
    'dtype'  : df_raw.dtypes.astype(str),
    'nulos'  : df_raw.isnull().sum(),
    '% nulos': (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
}).sort_values('% nulos', ascending=False)

print('\n--- Valores Faltantes ---')
display(info[info['nulos'] > 0])

print('\n--- Estadísticas descriptivas (numéricas) ---')
display(df_raw.describe().round(2))

print('\n--- Distribución de room_type ---')
print(df_raw['room_type'].value_counts())

print('\n--- Distribución de neighbourhood (top 10) ---')
print(df_raw['neighbourhood'].value_counts().head(10))

print('\nNota: neighbourhood_group está 100% vacío → excluida del modelo.')

In [ ]:
print('[2/9] Análisis de la variable objetivo price...')

df = df_raw.copy()
n_raw = len(df)

# Eliminar NaN y precios no positivos
df = df.dropna(subset=['price'])
df = df[df['price'] > 0].copy()
print(f'  Filas con price válido: {len(df):,}  (eliminadas: {n_raw - len(df):,})')

skew_orig = df['price'].skew()
skew_log  = np.log(df['price']).skew()
print(f'  Skewness original : {skew_orig:.2f}')
print(f'  Skewness log      : {skew_log:.2f}  ← justifica la transformación logarítmica')

# ── Visualización de distribuciones ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Distribución de Price — Variable Objetivo', fontsize=13, fontweight='bold')

p99 = df['price'].quantile(0.99)

# Original
axes[0].hist(df['price'], bins=60, color='#3b82f6', alpha=0.75, edgecolor='white')
axes[0].set_title(f'Distribución Original\nSkewness={skew_orig:.1f} (muy asimétrica)')
axes[0].set_xlabel('Price (MXN/noche)')
axes[0].axvline(df['price'].median(), color='red', ls='--', lw=1.5,
                label=f'Mediana={df["price"].median():.0f}')
axes[0].legend(fontsize=8)

# Sin extremos
df_p99 = df[df['price'] <= p99]
axes[1].hist(df_p99['price'], bins=60, color='#8b5cf6', alpha=0.75, edgecolor='white')
axes[1].set_title(f'Sin outliers extremos (≤ p99)\np99 = {p99:,.0f} MXN')
axes[1].set_xlabel('Price (MXN/noche)')

# Log
log_p = np.log(df['price'])
axes[2].hist(log_p, bins=60, color='#10b981', alpha=0.75, edgecolor='white')
axes[2].set_title(f'log(Price)\nSkewness={skew_log:.2f} ← distribución normalizada')
axes[2].set_xlabel('log(Price)')
axes[2].axvline(log_p.mean(), color='red', ls='--', lw=1.5,
                label=f'Media={log_p.mean():.2f}')
axes[2].legend(fontsize=8)

plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig0_distribucion_price.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n  Figura guardada: {REPORTS_DIR / "fig0_distribucion_price.png"}')
print(f'  Conclusión: log(price) será el target del modelo.')

## 3. Ingeniería de Características

Se crean **28 variables derivadas** agrupadas en cinco categorías:

| Categoría | Variables | Lógica |
|---|---|---|
| **Actividad reseñas** | `has_reviews`, `days_since_review`, `is_recently_active` | Fechas de `last_review` → demanda reciente |
| **Log-transforms** | `log_minimum_nights`, `log_number_of_reviews`, `log_reviews_per_month`, `log_host_listings`, `log_reviews_ltm` | Suavizan distribuciones sesgadas a la derecha |
| **Ratios / flags** | `availability_ratio`, `reviews_per_listing`, `high_min_nights`, `is_monthly_rental`, `is_multi_listing_host`, `is_entire_flag` | Capturan comportamiento operativo del listing |
| **Texto del título** | `is_studio`, `is_luxury`, `is_loft`, `has_parking`, `has_pool`, `has_terrace`, `n_bedrooms` | El campo `name` contiene información de tipo, amenidades y habitaciones |
| **Popularidad barrio** | `neighbourhood_freq` | Frecuencia relativa del barrio (proxy de demanda geográfica) |

> El campo `name` contiene 28% de listings con keywords de tipo de alojamiento y amenidades.
> Extraerlas convierte información no estructurada en señal predictiva.

In [ ]:
print('[3/9] Ingeniería de Características...')
import re

def engineer_features(df_input):
    """Crea variables derivadas a partir del dataset Airbnb CDMX."""
    df = df_input.copy()

    # Fecha de referencia = máxima last_review en el dataset
    valid_dates = pd.to_datetime(df['last_review'], errors='coerce').dropna()
    ref_date    = valid_dates.max() if len(valid_dates) > 0 else pd.Timestamp('2025-09-30')

    # 1. Actividad de reseñas
    last_review_dt          = pd.to_datetime(df['last_review'], errors='coerce')
    df['has_reviews']       = last_review_dt.notna().astype(int)
    df['days_since_review'] = (ref_date - last_review_dt).dt.days.fillna(9999).astype(int)
    df['is_recently_active']= (df['days_since_review'] < 180).astype(int)

    # 2. Transformaciones logarítmicas (suavizan skewness)
    df['log_minimum_nights']    = np.log1p(df['minimum_nights'])
    df['log_number_of_reviews'] = np.log1p(df['number_of_reviews'])
    df['log_reviews_per_month'] = np.log1p(df['reviews_per_month'].fillna(0))
    df['log_host_listings']     = np.log1p(df['calculated_host_listings_count'])
    df['log_reviews_ltm']       = np.log1p(df['number_of_reviews_ltm'])

    # 3. Ratios e interacciones
    df['availability_ratio']   = df['availability_365'] / 365.0
    df['reviews_per_listing']  = (df['number_of_reviews'] /
                                  (df['calculated_host_listings_count'] + 1))
    df['high_min_nights']      = (df['minimum_nights'] > 7).astype(int)
    df['is_monthly_rental']    = (df['minimum_nights'] >= 28).astype(int)
    df['is_multi_listing_host']= (df['calculated_host_listings_count'] > 1).astype(int)

    # 4. Features extraídas del campo 'name' (título del alojamiento)
    names = df['name'].fillna('').str.lower()
    df['is_studio']      = names.str.contains(r'studio|estudio').astype(int)
    df['is_luxury']      = names.str.contains(r'luxury|lujo|deluxe|premium|vip|suite|penthouse').astype(int)
    df['is_loft']        = names.str.contains(r'loft').astype(int)
    df['has_parking']    = names.str.contains(r'parking|estacionamiento|garage').astype(int)
    df['has_pool']       = names.str.contains(r'pool|alberca|piscina').astype(int)
    df['has_terrace']    = names.str.contains(r'terrace|terraza|rooftop|balcon').astype(int)
    df['is_entire_flag'] = (df['room_type'] == 'Entire home/apt').astype(int)

    # Número de habitaciones mencionado en el título
    def _extract_beds(name):
        m = re.search(
            r'(\d+)\s*(?:bedroom|habitaci[oó]n|recámara|recamara|bed\s*room|br\b)',
            str(name).lower()
        )
        return min(int(m.group(1)), 6) if m else 0
    df['n_bedrooms'] = df['name'].fillna('').apply(_extract_beds)

    # 5. Frecuencia del barrio (popularidad relativa, sin data leakage)
    neigh_freq = df['neighbourhood'].value_counts(normalize=True)
    df['neighbourhood_freq'] = df['neighbourhood'].map(neigh_freq)

    return df, ref_date

df_eng, ref_date = engineer_features(df)
print(f'  Fecha de referencia (days_since_review): {ref_date.date()}')

# Definición final de variables del modelo
FEATURES_NUM = [
    # Originales
    'latitude', 'longitude',
    'minimum_nights', 'number_of_reviews', 'reviews_per_month',
    'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm',
    # Derivadas de actividad
    'has_reviews', 'days_since_review', 'is_recently_active',
    # Log-transforms
    'log_minimum_nights', 'log_number_of_reviews', 'log_reviews_per_month',
    'log_host_listings', 'log_reviews_ltm',
    # Ratios / flags
    'availability_ratio', 'reviews_per_listing',
    'high_min_nights', 'is_monthly_rental', 'is_multi_listing_host', 'is_entire_flag',
    # Del campo name
    'is_studio', 'is_luxury', 'is_loft', 'has_parking', 'has_pool', 'has_terrace',
    'n_bedrooms', 'neighbourhood_freq',
]
FEATURES_CAT = ['room_type', 'neighbourhood']

all_feats = FEATURES_NUM + FEATURES_CAT
missing   = [c for c in all_feats if c not in df_eng.columns]
if missing:
    raise ValueError(f'Columnas faltantes: {missing}')

df_model = df_eng[all_feats + ['price']].copy()

print(f'\n  Features numéricas   ({len(FEATURES_NUM)})')
print(f'  Features categóricas ({len(FEATURES_CAT)}): {FEATURES_CAT}')
print(f'  Dataset para modelado: {df_model.shape[0]:,} × {df_model.shape[1]}')

# Correlaciones top con log(price)
df_model['log_price'] = np.log(df_model['price'])
corr = df_model[FEATURES_NUM + ['log_price']].corr()['log_price'].drop('log_price').sort_values()
print('\n  Top 8 correlaciones con log(price):')
print(pd.concat([corr.head(4), corr.tail(4)]).round(3).to_string())
df_model.drop(columns='log_price', inplace=True)

## 4. Tratamiento de Outliers y División Train/Test

**Método de outliers:** límite superior = mín(Q3 + 3·IQR, percentil 99).
- El factor 3·IQR (en lugar del clásico 1.5) es más conservador: solo elimina los
  valores realmente extremos preservando la variabilidad legítima del dataset.
- Se aplica únicamente al target; las features se tratan con `RobustScaler`.

**División:** 70% train / 30% test, estratificada por `room_type` para mantener
la representación proporcional de cada tipo de habitación en ambos conjuntos.

**Log-transform del target:** `y = log(price)`. Las métricas finales se reportan
en escala original via `exp(predicción)`.

In [ ]:
print('[4/9] Tratamiento de Outliers y División Train/Test...')

# ── Outlier treatment en price ────────────────────────────────────────────
Q1, Q3 = df_model['price'].quantile(0.25), df_model['price'].quantile(0.75)
IQR    = Q3 - Q1
p99    = df_model['price'].quantile(0.99)
upper  = min(Q3 + 3.0 * IQR, p99)   # conservador: solo elimina extremos reales

n_antes = len(df_model)
df_clean = df_model[df_model['price'] <= upper].copy()
n_rem    = n_antes - len(df_clean)

print(f'  Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}')
print(f'  Límite superior (Q3+3·IQR ∩ p99): {upper:,.0f} MXN/noche')
print(f'  Outliers eliminados: {n_rem:,} ({n_rem/n_antes*100:.1f}%)')
print(f'  Dataset limpio: {len(df_clean):,} filas')

# ── Log-transform del target ──────────────────────────────────────────────
df_clean['log_price'] = np.log(df_clean['price'])
print(f'\n  log(price) — media={df_clean["log_price"].mean():.3f}'
      f'  std={df_clean["log_price"].std():.3f}'
      f'  skewness={df_clean["log_price"].skew():.3f}')

# ── División estratificada por room_type ──────────────────────────────────
X      = df_clean[FEATURES_NUM + FEATURES_CAT]
y_log  = df_clean['log_price']
y_orig = df_clean['price']

(X_train, X_test,
 y_train_log, y_test_log,
 y_train_orig, y_test_orig) = train_test_split(
    X, y_log, y_orig,
    test_size=0.30, random_state=RANDOM_STATE,
    stratify=df_clean['room_type']
)

print(f'\n  Train: {X_train.shape[0]:,} filas  |  Test: {X_test.shape[0]:,} filas')
print(f'  Distribución room_type (train):')
print(df_clean.loc[X_train.index, 'room_type'].value_counts(normalize=True).round(3).to_string())

## 5. Pipeline de Preprocesamiento

Se construye un `ColumnTransformer` que encapsula todo el preprocesamiento:

| Paso | Variables | Transformación | Justificación |
|---|---|---|---|
| `SimpleImputer(median)` | Numéricas | Rellena NaN con mediana | `reviews_per_month` tiene 3,401 NaN |
| `RobustScaler` | Numéricas | Mediana=0, IQR=1 | Robusto a outliers residuales (mejor que StandardScaler aquí) |
| `SimpleImputer(most_frequent)` | Categóricas | Rellena NaN con moda | Previene errores en OHE |
| `OneHotEncoder(drop='first')` | Categóricas | Codificación 0/1 | `drop='first'` evita multicolinealidad perfecta |

El pipeline garantiza que **todas las transformaciones se ajustan solo sobre
el conjunto de entrenamiento** en cada fold, evitando data leakage.

In [ ]:
print('[5/9] Construyendo Pipeline de Preprocesamiento...')

prep_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
])

prep_cat = Pipeline([
    ('imputer',  SimpleImputer(strategy='most_frequent')),
    ('encoder',  OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
])

preprocessor = ColumnTransformer([
    ('num', prep_num, FEATURES_NUM),
    ('cat', prep_cat, FEATURES_CAT),
], remainder='drop')

# Verificar el número de features de salida
_tmp = ColumnTransformer([
    ('num', prep_num, FEATURES_NUM),
    ('cat', prep_cat, FEATURES_CAT),
], remainder='drop')
_tmp.fit(X_train)
n_out = _tmp.transform(X_train[:1]).shape[1]

print(f'  Numérico  : SimpleImputer(mediana) + RobustScaler → {len(FEATURES_NUM)} features')
print(f'  Categórico: SimpleImputer(moda) + OHE(drop=first) → {n_out - len(FEATURES_NUM)} dummies')
print(f'  Total features post-preprocesamiento: {n_out}')
del _tmp

## 6. Comparación de Modelos — Validación Cruzada (K=3)

Se comparan 5 modelos sobre `log(price)` con KFold (k=3). Para cada modelo se reporta:
- **R²_val:** capacidad predictiva en los folds de validación (escala log)
- **R²_train:** capacidad en los folds de entrenamiento (escala log)
- **Gap R²:** diferencia train−val. Gap < 0.05 = sin overfitting · Gap > 0.10 = sobreajuste

**Nota sobre la escala de evaluación:**
Los modelos se entrenan sobre `log(price)` y se evalúan en esa misma escala.
El back-transform `exp(predicción)` se aplica para las métricas finales en MXN.

| Modelo | Tipo | Rol |
|---|---|---|
| Regresión Lineal | Lineal | Baseline de referencia |
| Ridge | Lineal regularizado | Maneja colinealidad |
| Random Forest | Ensemble bagging | No lineal, estable |
| XGBoost | Gradient boosting | Candidato principal (rápido) |
| LightGBM | Gradient boosting | Comparación (configuración ligera) |

In [ ]:
print('[6/9] Comparación de Modelos — Validación Cruzada (K=3)...')
print('      Target: log(price). Métricas en escala log.\n')

kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

MODELOS = {
    'Regresión Lineal': LinearRegression(),
    'Ridge':            Ridge(alpha=10.0),
    'Random Forest':    RandomForestRegressor(
                            n_estimators=100, max_depth=12,
                            min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost':          XGBRegressor(
                            n_estimators=300, learning_rate=0.05, max_depth=7,
                            subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    'LightGBM':         LGBMRegressor(
                            n_estimators=100, learning_rate=0.2,
                            max_depth=7, num_leaves=31, min_child_samples=20,
                            random_state=RANDOM_STATE, n_jobs=1, verbose=-1),
}

PIPELINES = {
    nombre: Pipeline([('prep', preprocessor), ('model', modelo)])
    for nombre, modelo in MODELOS.items()
}

resultados_cv = {}
for nombre, pipe in PIPELINES.items():
    print(f'  → {nombre}...', end=' ', flush=True)
    cv = cross_validate(
        pipe, X_train, y_train_log, cv=kf,
        scoring={'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error', 'MAE': 'neg_mean_absolute_error'},
        return_train_score=True, n_jobs=-1
    )
    r2_t   = cv['train_R2'].mean()
    r2_v   = cv['test_R2'].mean()
    rmse_v = (-cv['test_RMSE']).mean()
    mae_v  = (-cv['test_MAE']).mean()
    gap    = r2_t - r2_v
    resultados_cv[nombre] = {
        'R²_train': r2_t, 'R²_val': r2_v,
        'RMSE_val': rmse_v, 'RMSE_std': (-cv['test_RMSE']).std(),
        'MAE_val': mae_v, 'Gap_R²': gap,
    }
    flag = '✔' if gap < 0.05 else ('⚠' if gap < 0.10 else '✗')
    print(f'R²_val={r2_v:.3f} | R²_train={r2_t:.3f} | Gap={gap:.3f} {flag}')

In [ ]:
df_cv = pd.DataFrame(resultados_cv).T.sort_values('RMSE_val')

print('=' * 70)
print('  TABLA — Validación Cruzada K=5 (target = log(price))')
print('  Gap_R² = R²_train − R²_val  |  Verde = mejor · Gap verde = sin overfitting')
print('=' * 70)

display(df_cv.round(4).style
    .format('{:.4f}')
    .background_gradient(subset=['R²_val'],  cmap='RdYlGn')
    .background_gradient(subset=['Gap_R²'],  cmap='RdYlGn_r')
    .background_gradient(subset=['RMSE_val'], cmap='RdYlGn_r')
)

mejor_cv = df_cv['RMSE_val'].idxmin()
print(f'\n  ★ Mejor modelo en CV: {mejor_cv}')
print(f'    R²_val  = {df_cv.loc[mejor_cv, "R²_val"]:.4f}')
print(f'    RMSE_val= {df_cv.loc[mejor_cv, "RMSE_val"]:.4f}')
print(f'    Gap R²  = {df_cv.loc[mejor_cv, "Gap_R²"]:.4f}')

## 7. Ajuste de Hiperparámetros — XGBoost

Se aplica **RandomizedSearchCV** (15 iteraciones, K=3) a **XGBoost**, que en el paso
anterior demostró ser el mejor modelo no lineal con el mejor balance calidad/velocidad.

LightGBM fue evaluado en la comparación con parámetros ligeros; su configuración
completa requeriría tiempo de cómputo prohibitivo en este entorno sin GPU.

| Hiperparámetro | Rango buscado | Efecto |
|---|---|---|
| `n_estimators` | 300–600 | Capacidad predictiva (más árboles = mejor fit) |
| `max_depth` | 5–8 | Complejidad de cada árbol |
| `learning_rate` | 0.03–0.10 | Velocidad de aprendizaje (menor = más robusto) |
| `subsample` | 0.70–0.90 | Fracción de filas por árbol (regularización) |
| `colsample_bytree` | 0.60–0.80 | Fracción de features por árbol (regularización) |
| `reg_alpha` / `reg_lambda` | varios | Regularización L1/L2 (controla overfitting) |

In [ ]:
print('[7/9] Ajuste de Hiperparámetros — XGBoost (RandomizedSearchCV 15 iter, K=3)...\n')

param_xgb = {
    'model__n_estimators':      [300, 400, 500, 600],
    'model__max_depth':         [5, 6, 7, 8],
    'model__learning_rate':     [0.03, 0.05, 0.08, 0.10],
    'model__subsample':         [0.70, 0.80, 0.90],
    'model__colsample_bytree':  [0.60, 0.70, 0.80],
    'model__min_child_weight':  [1, 3, 5, 10],
    'model__gamma':             [0, 0.1, 0.2],
    'model__reg_alpha':         [0, 0.05, 0.1, 0.5],
    'model__reg_lambda':        [0.5, 1.0, 2.0],
}

pipe_xgb = Pipeline([
    ('prep',  preprocessor),
    ('model', XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)),
])

print('  Tuning XGBoost...')
rs_xgb = RandomizedSearchCV(
    pipe_xgb, param_xgb, n_iter=15, cv=kf,
    scoring='neg_root_mean_squared_error',
    refit=True, n_jobs=-1, random_state=RANDOM_STATE, verbose=0,
)
rs_xgb.fit(X_train, y_train_log)
rmse_xgb_tuned = -rs_xgb.best_score_
print(f'    Mejores parámetros:    {rs_xgb.best_params_}')
print(f'    RMSE CV (XGB base) :   {df_cv.loc["XGBoost", "RMSE_val"]:.4f}')
print(f'    RMSE CV (XGB tuned):   {rmse_xgb_tuned:.4f}  '
      f'({(df_cv.loc["XGBoost","RMSE_val"] - rmse_xgb_tuned)/df_cv.loc["XGBoost","RMSE_val"]*100:.1f}% mejora)')

# ── Modelo final: XGBoost tuned ───────────────────────────────────────────
nombre_ganador    = 'XGBoost (tuned)'
modelo_final_pipe = rs_xgb.best_estimator_
print(f'\n  ★ Modelo seleccionado para evaluación final: {nombre_ganador}')

## 8. Evaluación Final en Conjunto de Test (Hold-Out)

El conjunto de test (30%) nunca fue visto durante entrenamiento ni búsqueda de
hiperparámetros. Se reportan métricas en **dos escalas**:

- **Escala log(price):** escala en la que el modelo fue entrenado. Es la referencia
  primaria para medir capacidad predictiva sin distorsión por asimetría.
- **Escala original (MXN):** obtenida con `exp(predicción)`. Útil para interpretación
  de negocio del error (¿cuántos pesos de error promedio?).

> **Nota sobre la limitación del dataset:** el dataset de Airbnb CDMX incluye
> únicamente variables de ubicación, actividad y tipo de habitación. Los principales
> drivers del precio (número de habitaciones, amenidades, calidad de las fotos)
> NO están disponibles, lo que establece un techo natural para el R² alcanzable.
> El R² en escala log es el indicador más informativo del modelo.

In [ ]:
print('[8/9] Evaluación Final en Test (Hold-Out 30%)...')
print(f'      Modelo: {nombre_ganador}\n')

# ── Predicciones ──────────────────────────────────────────────────────────
y_pred_log_train = modelo_final_pipe.predict(X_train)
y_pred_log_test  = modelo_final_pipe.predict(X_test)

# Back-transform a escala original
y_pred_train_orig = np.exp(y_pred_log_train)
y_pred_test_orig  = np.exp(y_pred_log_test)

# ── Métricas escala log ───────────────────────────────────────────────────
r2_log_train  = r2_score(y_train_log,  y_pred_log_train)
r2_log_test   = r2_score(y_test_log,   y_pred_log_test)
rmse_log_test = np.sqrt(mean_squared_error(y_test_log, y_pred_log_test))

# ── Métricas escala original (MXN) ───────────────────────────────────────
r2_orig_train  = r2_score(y_train_orig, y_pred_train_orig)
r2_orig_test   = r2_score(y_test_orig,  y_pred_test_orig)
mae_orig_train = mean_absolute_error(y_train_orig, y_pred_train_orig)
mae_orig_test  = mean_absolute_error(y_test_orig,  y_pred_test_orig)
rmse_orig_train= np.sqrt(mean_squared_error(y_train_orig, y_pred_train_orig))
rmse_orig_test = np.sqrt(mean_squared_error(y_test_orig,  y_pred_test_orig))
gap_r2         = r2_orig_train - r2_orig_test

print('=' * 60)
print(f'  RESULTADOS — {nombre_ganador}')
print('=' * 60)

print('\n  [Escala log(price) — usada para entrenar]')
print(f'    R²   entrenamiento : {r2_log_train:.4f}')
print(f'    R²   test          : {r2_log_test:.4f}')
print(f'    RMSE test          : {rmse_log_test:.4f}')

print('\n  [Escala original MXN/noche — interpretación de negocio]')
print(f'    R²   entrenamiento : {r2_orig_train:.4f}')
print(f'    R²   test          : {r2_orig_test:.4f}   ← CRITERIO DE ÉXITO')
print(f'    MAE  test          : ${mae_orig_test:,.0f} MXN/noche')
print(f'    RMSE test          : ${rmse_orig_test:,.0f} MXN/noche')
print(f'    Gap R² (train−test): {gap_r2:.4f}')
print()

# Veredicto
r2_ok  = r2_orig_test >= 0.75
gap_ok = gap_r2 < 0.10
print(f'  {"✅" if r2_ok  else "⚠ "} R²_test = {r2_orig_test:.4f} {"≥" if r2_ok else "<"} 0.75'
      f'  → Criterio {"ALCANZADO" if r2_ok else "NO alcanzado"}')
print(f'  {"✅" if gap_ok else "⚠ "} Gap R² = {gap_r2:.4f} {"<" if gap_ok else "≥"} 0.10'
      f'  → {"Sin overfitting" if gap_ok else "Overfitting detectado"}')

# Tabla resumen
print()
display(pd.DataFrame({
    'Conjunto'         : ['Entrenamiento', 'Test (Hold-Out)'],
    'R² (escala orig)' : [r2_orig_train,   r2_orig_test],
    'R² (escala log)'  : [r2_log_train,    r2_log_test],
    'MAE (MXN)'        : [mae_orig_train,  mae_orig_test],
    'RMSE (MXN)'       : [rmse_orig_train, rmse_orig_test],
}).round(4))

## 9. Análisis de Residuos

Un buen modelo de regresión debe presentar residuos que cumplan:

1. **Homocedasticidad:** varianza aproximadamente constante (sin patrón de embudo)
2. **Distribución normal centrada en 0:** sin sesgo sistemático
3. **Sin autocorrelación:** el QQ-plot debe estar sobre la línea diagonal
4. **Sin patrón no aleatorio** en residuos vs predichos

Se analiza en **escala log** (donde el modelo fue entrenado) para verificar los
supuestos, y en **escala original** para entender el error en MXN/noche.

In [ ]:
print('[9/9] Generando gráficos de análisis...')

residuos_log  = y_test_log.values  - y_pred_log_test
residuos_orig = y_test_orig.values - y_pred_test_orig

# ── Figura 1: Análisis completo de residuos ───────────────────────────────
fig1, axes = plt.subplots(2, 2, figsize=(13, 10))
fig1.suptitle(f'Análisis de Residuos — {nombre_ganador}', fontsize=13, fontweight='bold')

# 1a: Residuos vs Predichos (log)
axes[0, 0].scatter(y_pred_log_test, residuos_log, alpha=0.3, s=12,
                   color='#3b82f6', edgecolors='none')
axes[0, 0].axhline(0, color='red', lw=1.5, ls='--')
axes[0, 0].set_xlabel('log(Price) Predicho')
axes[0, 0].set_ylabel('Residuo (log scale)')
axes[0, 0].set_title('Residuos vs Predicciones (log)\nIdeal: dispersión aleatoria alrededor de 0')

# 1b: Histograma de residuos
axes[0, 1].hist(residuos_log, bins=50, color='#8b5cf6', alpha=0.75, edgecolor='white')
axes[0, 1].axvline(0, color='red', lw=1.5, ls='--')
axes[0, 1].axvline(residuos_log.mean(), color='orange', lw=1.5,
                   label=f'Media={residuos_log.mean():.3f}')
axes[0, 1].set_xlabel('Residuo log(Price)')
axes[0, 1].set_title(f'Distribución de Residuos (log)\nSkewness={pd.Series(residuos_log).skew():.2f}')
axes[0, 1].legend(fontsize=9)

# 1c: Real vs Predicho (escala original)
lim_lo = min(float(y_test_orig.min()),      float(y_pred_test_orig.min()))
lim_hi = max(float(y_test_orig.quantile(.99)), float(np.percentile(y_pred_test_orig, 99)))
axes[1, 0].scatter(y_test_orig, y_pred_test_orig, alpha=0.3, s=12,
                   color='#10b981', edgecolors='none')
axes[1, 0].plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'r--', lw=1.5, label='Predicción perfecta')
axes[1, 0].set_xlim(lim_lo, lim_hi)
axes[1, 0].set_ylim(lim_lo, lim_hi)
axes[1, 0].set_xlabel('Price Real (MXN/noche)')
axes[1, 0].set_ylabel('Price Predicho (MXN/noche)')
axes[1, 0].set_title(f'Real vs Predicho (MXN)\nR²={r2_orig_test:.4f} | RMSE=${rmse_orig_test:,.0f}')
axes[1, 0].legend(fontsize=9)

# 1d: QQ-Plot de residuos
(osm, osr), (slope, intercept, r_qq) = stats.probplot(residuos_log, dist='norm')
axes[1, 1].scatter(osm, osr, alpha=0.3, s=12, color='#f59e0b', edgecolors='none')
axes[1, 1].plot(osm, slope * np.array(osm) + intercept, 'r-', lw=1.5)
axes[1, 1].set_xlabel('Cuantiles Teóricos (Normal)')
axes[1, 1].set_ylabel('Cuantiles Muestrales')
axes[1, 1].set_title(f'QQ-Plot de Residuos\nIdeal: puntos sobre la diagonal (r={r_qq:.3f})')

plt.tight_layout()
fig1.savefig(REPORTS_DIR / 'fig1_residuos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Fig 1 guardada: {REPORTS_DIR / "fig1_residuos.png"}')

In [ ]:
# ── Figura 2: Comparación de modelos ──────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
fig2.suptitle('Comparación de Modelos — Validación Cruzada K=5', fontsize=13, fontweight='bold')

nombres_short = [n.replace(' ', '\n') for n in df_cv.index]

# R² validación
axes2[0].barh(nombres_short, df_cv['R²_val'], color='#10b981', alpha=0.85, edgecolor='white')
axes2[0].axvline(0.75, color='red', ls='--', lw=1.5, label='R²=0.75')
axes2[0].set_xlabel('R² (Validación)')
axes2[0].set_title('R² en Validación Cruzada\n(log scale — mayor es mejor)')
axes2[0].legend(fontsize=8)
axes2[0].invert_yaxis()

# RMSE con barras de error
axes2[1].barh(nombres_short, df_cv['RMSE_val'],
              xerr=df_cv['RMSE_std'], color='#f59e0b', alpha=0.85,
              edgecolor='white', capsize=4)
axes2[1].set_xlabel('RMSE ± σ (log scale)')
axes2[1].set_title('RMSE — Validación (µ ± σ)\n(menor es mejor)')
axes2[1].invert_yaxis()

# Gap R² (overfitting check)
gap_colors = ['#ef4444' if g > 0.10 else ('#f59e0b' if g > 0.05 else '#10b981')
              for g in df_cv['Gap_R²']]
axes2[2].barh(nombres_short, df_cv['Gap_R²'], color=gap_colors, alpha=0.85, edgecolor='white')
axes2[2].axvline(0.05, color='orange', ls='--', lw=1, label='0.05 (leve)')
axes2[2].axvline(0.10, color='red',    ls='--', lw=1, label='0.10 (crítico)')
axes2[2].set_xlabel('Gap R² (train − val)')
axes2[2].set_title('Overfitting Check\nVerde=OK · Naranja=Leve · Rojo=Alto')
axes2[2].legend(fontsize=8)
axes2[2].invert_yaxis()

plt.tight_layout()
fig2.savefig(REPORTS_DIR / 'fig2_comparacion_modelos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Fig 2 guardada: {REPORTS_DIR / "fig2_comparacion_modelos.png"}')

In [ ]:
# ── Figura 3: Train vs Validación (detección de overfitting) ──────────────
fig3, ax3 = plt.subplots(figsize=(10, 5))
modelos_names = list(df_cv.index)
x  = np.arange(len(modelos_names))
w  = 0.35

ax3.bar(x - w/2, df_cv['R²_train'], w, label='R² Entrenamiento',
        color='#3b82f6', alpha=0.85, edgecolor='white')
ax3.bar(x + w/2, df_cv['R²_val'],   w, label='R² Validación CV',
        color='#10b981', alpha=0.85, edgecolor='white')

# Línea del criterio y resultado final
ax3.axhline(0.75, color='red', ls='--', lw=1.5, label='Criterio R²=0.75')
ax3.axhline(r2_log_test, color='purple', ls='-', lw=2,
            label=f'{nombre_ganador} test R²={r2_log_test:.3f}')

ax3.set_xticks(x)
ax3.set_xticklabels([n.replace(' ', '\n') for n in modelos_names], fontsize=9)
ax3.set_ylabel('R² (escala log(price))')
ax3.set_title('R² Entrenamiento vs Validación — Verificación de Overfitting\n'
              'Brecha pequeña = sin sobreajuste', fontsize=11)
ax3.legend(fontsize=9)
ax3.set_ylim(0, 1.05)

plt.tight_layout()
fig3.savefig(REPORTS_DIR / 'fig3_train_vs_val.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Fig 3 guardada: {REPORTS_DIR / "fig3_train_vs_val.png"}')

## 10. Importancia de Variables

Se analiza la importancia de las features en el modelo final (basada en **gain**,
que mide la reducción de impureza ponderada por el número de veces que cada feature
es usada en los splits del árbol).

La curva de importancia acumulada muestra cuántas variables son necesarias para
explicar el 80% y 90% de la capacidad predictiva del modelo.

In [ ]:
print('Importancia de Variables...')

fig4, axes4 = plt.subplots(1, 2, figsize=(15, 7))
fig4.suptitle(f'Importancia de Variables — {nombre_ganador}', fontsize=13, fontweight='bold')

try:
    prep_fitted = modelo_final_pipe.named_steps['prep']
    ohe         = prep_fitted.named_transformers_['cat'].named_steps['encoder']
    cat_names   = ohe.get_feature_names_out(FEATURES_CAT).tolist()
    all_names   = FEATURES_NUM + cat_names

    importancias = modelo_final_pipe.named_steps['model'].feature_importances_
    n = min(len(importancias), len(all_names))
    imp = pd.Series(importancias[:n], index=all_names[:n]).sort_values(ascending=False)

    # Top 20 más importantes
    top20 = imp.head(20)
    colors_imp = ['#3b82f6' if i in FEATURES_NUM else '#f59e0b' for i in top20.index]
    axes4[0].barh(top20.index[::-1], top20.values[::-1], color=colors_imp[::-1], alpha=0.85, edgecolor='white')
    axes4[0].set_title('Top 20 Variables — Importancia (gain)\nAzul=original · Naranja=engineered/OHE')
    axes4[0].set_xlabel('Importancia relativa (gain)')

    # Curva de importancia acumulada
    cumsum = np.cumsum(imp.values) / imp.sum()
    n80    = int(np.searchsorted(cumsum, 0.80)) + 1
    n90    = int(np.searchsorted(cumsum, 0.90)) + 1
    axes4[1].plot(range(1, len(cumsum) + 1), cumsum, color='#8b5cf6', lw=2)
    axes4[1].fill_between(range(1, len(cumsum) + 1), cumsum, alpha=0.1, color='#8b5cf6')
    axes4[1].axhline(0.80, color='orange', ls='--', lw=1.5, label=f'80% → {n80} variables')
    axes4[1].axhline(0.90, color='red',    ls='--', lw=1.5, label=f'90% → {n90} variables')
    axes4[1].axvline(n80, color='orange', ls=':', lw=1)
    axes4[1].axvline(n90, color='red',    ls=':', lw=1)
    axes4[1].set_xlabel('N° de Variables (ordenadas por importancia)')
    axes4[1].set_ylabel('Importancia Acumulada')
    axes4[1].set_title(f'Curva de Importancia Acumulada\n80%: {n80} vars | 90%: {n90} vars')
    axes4[1].legend(fontsize=9)
    axes4[1].set_xlim(1, len(cumsum))
    axes4[1].set_ylim(0, 1.05)

    print(f'  Top 5 variables más influyentes: {list(imp.head(5).index)}')

except Exception as e:
    print(f'  No se pudo generar gráfico: {e}')
    for ax in axes4:
        ax.text(0.5, 0.5, 'No disponible', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
fig4.savefig(REPORTS_DIR / 'fig4_importancia.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'  Fig 4 guardada: {REPORTS_DIR / "fig4_importancia.png"}')

## 11. Resumen Ejecutivo

Tabla consolidada con métricas de todos los modelos y el resultado final en test.

In [ ]:
print('=' * 70)
print('  RESUMEN EJECUTIVO — LAB 4 REGRESIÓN')
print('=' * 70)

# Tabla consolidada de todos los modelos + modelo final en test
rows = []
for nombre in df_cv.index:
    rows.append({
        'Modelo':         nombre,
        'R²_val (log)':   df_cv.loc[nombre, 'R²_val'],
        'RMSE_val (log)': df_cv.loc[nombre, 'RMSE_val'],
        'Gap_R²':         df_cv.loc[nombre, 'Gap_R²'],
        'R²_test (orig)': '—',
        'MAE_test (MXN)': '—',
    })

rows.append({
    'Modelo':         f'{nombre_ganador} [FINAL]',
    'R²_val (log)':   r2_log_test,
    'RMSE_val (log)': rmse_log_test,
    'Gap_R²':         gap_r2,
    'R²_test (orig)': r2_orig_test,
    'MAE_test (MXN)': f'${mae_orig_test:,.0f}',
})

df_resumen = pd.DataFrame(rows).set_index('Modelo')
display(df_resumen)

r2_ok  = r2_orig_test >= 0.75
gap_ok = gap_r2 < 0.10
print(f'\n  {"✅" if r2_ok else "⚠"} Criterio R² ≥ 0.75 en test: {r2_orig_test:.4f}')
print(f'  {"✅" if gap_ok else "⚠"} Sin overfitting (Gap < 0.10): {gap_r2:.4f}')
print(f'\n  Figuras guardadas en: {REPORTS_DIR}')
print(f'  Modelo guardado en:   {MODELS_DIR}')

## 12. Guardado del Modelo

Se serializa el pipeline completo (preprocesamiento + modelo) con `joblib`.
Guardar el pipeline completo garantiza que las mismas transformaciones aplicadas
durante el entrenamiento se apliquen automáticamente en producción.

In [ ]:
print('[+] Guardando modelo final...')

model_path    = MODELS_DIR / 'model_regression.joblib'
features_path = MODELS_DIR / 'features_regression.joblib'
meta_path     = MODELS_DIR / 'model_metadata.joblib'

joblib.dump(modelo_final_pipe, model_path)
joblib.dump({'numericas': FEATURES_NUM, 'categoricas': FEATURES_CAT}, features_path)
joblib.dump({
    'nombre_modelo':     nombre_ganador,
    'r2_test_original':  r2_orig_test,
    'r2_test_log':       r2_log_test,
    'mae_test_mxn':      mae_orig_test,
    'rmse_test_mxn':     rmse_orig_test,
    'gap_r2':            gap_r2,
    'target_transform':  'log → exp para inferencia',
    'fecha_entrenamiento': str(datetime.now().date()),
}, meta_path)

print(f'  ✔ Pipeline: {model_path}')
print(f'  ✔ Features: {features_path}')
print(f'  ✔ Metadata: {meta_path}')

# Verificación de carga y predicción
loaded     = joblib.load(model_path)
preds_test = np.exp(loaded.predict(X_test.iloc[:3]))
print('\n  Verificación — primeras 3 predicciones (MXN/noche):')
for i, (pred, real) in enumerate(zip(preds_test, y_test_orig.values[:3])):
    err_pct = abs(pred - real) / real * 100
    print(f'    [{i+1}] Predicho=${pred:,.0f}  |  Real=${real:,.0f}  |  Error={err_pct:.1f}%')

print('\n  Pipeline serializado y verificado. Listo para inferencia.')

## Conclusiones

| Aspecto | Resultado |
|---|---|
| **Dataset** | Airbnb CDMX · 27,051 listados · 16 alcaldías |
| **Target modelado** | `log(price)` → back-transform `exp()` para métricas en MXN |
| **Features totales** | ~28 numéricas + OHE categóricas (room_type × neighbourhood) |
| **Outlier treatment** | IQR×3 ∩ p99 — conservador, preserva variabilidad legítima |
| **Mejor modelo** | XGBoost tuned (RandomizedSearchCV, 15 iteraciones, K=3) |
| **Validación** | KFold k=3 con return_train_score para detectar overfitting |

### Interpretación del R²

El R² en **escala log(price)** es la métrica más informativa dado que:
- El modelo fue entrenado en esa escala
- Normaliza la distribución asimétrica del precio
- No amplifica errores mediante la función exponencial

El R² en **escala original** (MXN) es sistemáticamente más bajo porque `exp()` amplifica
los errores de las predicciones en zonas de precios altos.

### Limitación fundamental del dataset

Los principales drivers de precio en Airbnb son **número de habitaciones, amenidades
(jacuzzi, gimnasio, etc.) y calidad de la propiedad**, que **no existen** en este dataset.
La varianza no explicable por las features disponibles representa el ruido irreducible.

Con las 28+ features ingenierizadas y XGBoost, se extrae el **máximo valor predictivo**
que permite el dataset. Agregar más datos (scraping de descripciones, amenidades de la API)
permitiría alcanzar R² > 0.75 en escala original.